# Test Robot Motion

Runs MuJoCo and loads the MJCF file to simulate the robot. Use the up arrow key to increase both motor speeds in the forward direction, and use the down arrow key to increase both motor speeds in the reverse direction. Try to balance the bot manually. Good luck!

MuJoCo API reference: [https://mujoco.readthedocs.io/en/stable/APIreference/index.html](https://mujoco.readthedocs.io/en/stable/APIreference/index.html)

In [1]:
# Import standard libraries
from pathlib import Path
import subprocess
import sys
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [2]:
# Settings
MJCF_PATH = Path.cwd().parents[1] / "mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml"
MOTOR_SPEED_STEP = 0.1     # How much to increase/decrease motor speed by
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 250          # Refresh the status display twice per second

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"
LEFT_WHEEL_POS = "left_wheel_pos"
RIGHT_WHEEL_POS = "right_wheel_pos"
LEFT_WHEEL_VEL = "left_wheel_vel"
RIGHT_WHEEL_VEL = "right_wheel_vel"

def format_sensor(values):
    return "[" + ", ".join(f"{value: .3f}" for value in values) + "]"

# Keys
KEY_BACKSPACE = 259
KEY_UP = 265
KEY_DOWN = 264

In [3]:
# Global: speed of each motor
ctrl = {"left": 0.0, "right": 0.0}

In [4]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [5]:
def key_callback(keycode):
    """Handle a key press"""
    # MuJoCo resets the sim with backspace, but we also need to stop the motors
    if keycode == KEY_BACKSPACE:
        ctrl["left"] = ctrl["right"] = 0.0
        return

    # For each up or down arrow press, increase or decrease both motor speeds
    if keycode == KEY_UP:
        ctrl["left"]  = clamp(ctrl["left"]  + MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] + MOTOR_SPEED_STEP)
    elif keycode == KEY_DOWN:
        ctrl["left"]  = clamp(ctrl["left"]  - MOTOR_SPEED_STEP)
        ctrl["right"] = clamp(ctrl["right"] - MOTOR_SPEED_STEP)
    else:
        return

In [6]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [7]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [8]:
if sys.platform == "darwin":
    mjpython = Path(sys.executable).with_name("mjpython")
    if not mjpython.is_file():
        raise FileNotFoundError(f"mjpython not found next to the active kernel: {mjpython}. Select the project .venv kernel.")
    script_path = Path.cwd() / "test_motion.py"
    subprocess.run([str(mjpython), str(script_path)], check=True)
else:
    # Resets simulation data to defaults
    mujoco.mj_resetData(model, data)

    # Launch MuJoCo simulator and GUI
    steps = 0
    with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
        # Define free-look camera (control with mouse), looking at robot's back-right
        viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        viewer.cam.lookat[:] = [0, 0, 0.05]
        viewer.cam.distance  = 0.8
        viewer.cam.azimuth   = 45
        viewer.cam.elevation = -25

        # Simulation loop
        while viewer.is_running():
            step_start = time.time()

            # Write the latest ctrl values into the sim.
            data.ctrl[left_motor_id] = ctrl["left"]
            data.ctrl[right_motor_id] = ctrl["right"]

            # Advance simulation by one step (determined by timestep attribute in MJCF file)
            mujoco.mj_step(model, data)

            # Render the current simulation state
            viewer.sync()

            # Refresh the compact sensor display twice per second
            steps += 1
            if steps % PRINT_EVERY == 0:
                accel = format_sensor(data.sensor(IMU_ACCEL).data)
                gyro = format_sensor(data.sensor(IMU_GYRO).data)
                orientation = format_sensor(data.sensor(IMU_ORIENTATION).data)
                left_pos = format_sensor(data.sensor(LEFT_WHEEL_POS).data)
                right_pos = format_sensor(data.sensor(RIGHT_WHEEL_POS).data)
                left_vel = format_sensor(data.sensor(LEFT_WHEEL_VEL).data)
                right_vel = format_sensor(data.sensor(RIGHT_WHEEL_VEL).data)

                clear_output(wait=True)
                print("MuJoCo: Manual motion test")
                print("Controls: Up/Down change both motors; Backspace stops them; close viewer to quit.")
                print("Readout refresh: 2 Hz | sensor values rounded to 3 decimals")
                print(f"Accel:          {accel}")
                print(f"Gyro:           {gyro}")
                print(f"Orientation:    {orientation}")
                print(f"Wheel position: L {left_pos}  R {right_pos}")
                print(f"Wheel velocity: L {left_vel}  R {right_vel}")
                print(f"Motor command:  L {ctrl['left']:+.1f}  R {ctrl['right']:+.1f}")

            # Keep simulation time close to real time
            slack = model.opt.timestep - (time.time() - step_start)
            if slack > 0:
                time.sleep(slack)

MuJoCo: Manual motion test
Model: /Users/sev1pi/vimal/git/experiments/reinforcement-learning-for-robotics/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml
Motor IDs: left 0, right 1
Controls: Up/Down change both motors; Backspace stops them; close viewer to quit.
Readout refresh: 2 Hz | sensor values rounded to 3 decimals

Accel:          [-0.000, -0.000,  9.810]
Gyro:           [-0.000, -0.000, -0.000]
Orientation:    [ 1.000, -0.000, -0.000, -0.000]
Wheel position: L [-0.000]  R [-0.000]
Wheel velocity: L [-0.000]  R [-0.000]
Motor command:  L +0.0  R +0.0
MuJoCo: Manual motion test
Model: /Users/sev1pi/vimal/git/experiments/reinforcement-learning-for-robotics/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml
Motor IDs: left 0, right 1
Controls: Up/Down change both motors; Backspace stops them; close viewer to quit.
Readout refresh: 2 Hz | sensor values rounded to 3 decimals

Accel:          [ 3.985, -0.040,  4.161]
Gyro:           [ 0.019, -8.091,  0